In [46]:
import duckdb
from pathlib import Path


project_root = Path.cwd().resolve()
while project_root.name != "big_data_assignment" and project_root.parent != project_root:
    project_root = project_root.parent

json_dir = project_root / "data" / "raw" / "json"
writers_json = json_dir / "writing.json"
directing_json = json_dir / "directing.json"


In [47]:
con = duckdb.connect()

df_writers = con.execute(f"""
SELECT *
FROM read_json_auto('{writers_json}')
""").df()

df_writers.head()

,movie,writer
0,tt0003740,nm0195339
1,tt0003740,nm0515385
2,tt0003740,nm0665163
3,tt0003740,nm0758215
4,tt0008663,nm0406585


# Problem: JSON is nested and we need a many-to-many
we want a junction table: one row per (movie, director).

### We need to explode the directers JSON

In [48]:

con = duckdb.connect(str(project_root / "members" / "lisa" / "notebooks" / "imdb.duckdb"))

In [49]:
con.execute(f"""
CREATE OR REPLACE TABLE movie_writer AS
SELECT movie, writer
FROM read_json_auto('{writers_json}');
""")

## Normalize the nested directers JSON 

In [59]:

con.execute(f"""
CREATE OR REPLACE TABLE movie_director AS
WITH src AS (
  SELECT
    json(movie)    AS movie_obj,
    json(director) AS director_obj
  FROM read_json_auto('{directing_json}')
),
movies AS (
  SELECT
    je.key AS k,
    trim(both '"' from CAST(je.value AS VARCHAR)) AS movie
  FROM src, json_each(movie_obj) je
),
directors AS (
  SELECT
    je.key AS k,
    trim(both '"' from CAST(je.value AS VARCHAR)) AS director
  FROM src, json_each(director_obj) je
)
SELECT
  m.movie,
  d.director
FROM movies m
JOIN directors d USING (k)
WHERE m.movie IS NOT NULL AND m.movie <> ''
  AND d.director IS NOT NULL AND d.director <> '';
""")

In [60]:
con.execute("SELECT * FROM movie_director LIMIT 10").df()
con.execute("SELECT COUNT(*) FROM movie_director").fetchone()

(11162,)

In [61]:
con.execute("SELECT * FROM movie_writer LIMIT 10").df()

,movie,writer
0,tt0003740,nm0195339
1,tt0003740,nm0515385
2,tt0003740,nm0665163
3,tt0003740,nm0758215
4,tt0008663,nm0406585
5,tt0008663,nm0596410
6,tt0008663,nm0803705
7,tt0009369,nm0215874
8,tt0009369,nm0370271
9,tt0010307,nm0304098


In [62]:
con.execute("SELECT * FROM movie_director LIMIT 10").df()

,movie,director
0,tt9911196,nm0631590
1,tt9904802,nm0052054
2,tt9900782,nm7992231
3,tt9850386,nm0550881
4,tt9850344,nm0284774
5,tt9849004,nm2818863
6,tt9845110,nm2889273
7,tt9840958,nm4672801
8,tt9831136,nm0261629
9,tt9820352,nm0053028
